# Zolai — master pipeline (Kaggle)

This notebook **coordinates** the full path: **clean → optional Gemini / Gemma → combine → audit → export → optional translation → train (separate notebook)**.

## Can everything be one notebook?

**Yes in one file**, but **not always one Run All session** on Kaggle:

- **CPU-heavy:** standard clean, combiner, gap audit, HF `save_to_disk` — use **CPU** accelerator to save GPU quota.
- **API calls:** Gemini clean / translation need **Internet** + API key (costs tokens).
- **GPU:** local **Gemma** and **Qwen LoRA** need **GPU**; both in one session are **long** and may hit **disk / time limits**.
- **Practical:** split into **2–3 sessions** (e.g. CPU pipeline → GPU Gemma → GPU Qwen) or use **flags** below to run only what you need.

## Canonical modular notebooks (same logic, easier to debug)

| Notebook | Role |
|----------|------|
| `zolai-cleaner-v2.ipynb` | Standard clean + optional Gemini + splits + HF export |
| `zolai-dataset-combiner.ipynb` | Merge many sources |
| `zolai-dataset-gap-audit.ipynb` | Post-clean stats + gap checklist |
| `zolai-cleaning-pipeline-local-gemma-2b.ipynb` | Local Gemma (GPU) |
| `zolai-qwen-training-v2.ipynb` | LoRA fine-tune (GPU) |

**Versioning:** keep **one** Kaggle master dataset and **bump versions** when you publish new cleaned files (see your earlier workflow).


## 1) One config — set flags per session

Edit `PIPELINE` and run the **plan** cell. Then open the **modular** notebooks listed above, or run **stage cells** in this notebook (sections 2–6) if present.


In [ ]:
import os
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("ZOLOAI_WORK_DIR", ".")).resolve()

# --- What to run in this session (toggle True/False) ---
PIPELINE = {
    "standard_clean": True,       # NFKC + dedupe + soft + noise (like cleaner)
    "gemini_clean": False,        # LLM fix + optional EN in cleaner (costs tokens)
    "local_gemma": False,         # GPU: run zolai-cleaning-pipeline-local-gemma-2b.ipynb separately
    "combine": False,             # merge sources: zolai-dataset-combiner.ipynb
    "gap_audit": False,           # stats + samples: zolai-dataset-gap-audit.ipynb
    "hf_export": True,            # DatasetDict save_to_disk after splits
    "gemini_translate_only": False,  # fill translation_en missing (API below)
    "qwen_train": False,          # GPU: run zolai-qwen-training-v2.ipynb separately
}


def print_plan():
    print("WORK_DIR:", WORK_DIR)
    print("--- Pipeline plan ---")
    for k, v in PIPELINE.items():
        print(f"  {k}: {v}")
    print("--- Recommended sessions ---")
    print("  Session A (CPU): standard_clean + optional gemini_clean + hf_export + gap_audit + combine")
    print("  Session B (GPU): local_gemma notebook (if you use Gemma)")
    print("  Session C (GPU): qwen_train notebook")


print_plan()


## 2) How to run modular notebooks (no duplicate code)

1. **Session A (CPU):** Run **`zolai-cleaner-v2.ipynb`** with `RUN_GEMINI_REFINE` from `PIPELINE` intent. Then **`zolai-dataset-combiner.ipynb`** if `combine=True`. Then **`zolai-dataset-gap-audit.ipynb`**.
2. **Session B (GPU):** Run **`zolai-cleaning-pipeline-local-gemma-2b.ipynb`** if you want local Gemma (not Gemini).
3. **Session C (GPU):** Run **`zolai-qwen-training-v2.ipynb`** with `ZOLOAI_DATASET_SRC` pointing at `.../zolai_hf_disk_export`.

Set env **`ZOLOAI_AUDIT_JSONL`**, **`ZOLOAI_DATASET_SRC`**, **`GEMINI_API_KEY`** / **`GOOGLE_API_KEY`** as needed.


## 3) Optional: Gemini translation pass (fill `translation_en`)

Use on a JSONL that already has **`text`** but missing or empty **`translation_en`**. Batched; costs API tokens. Set **`RUN_TRANSLATION = True`** in the next cell.


In [ ]:
import json
import os
import sys
import time
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else os.environ.get("ZOLOAI_WORK_DIR", ".")).resolve()
RUN_TRANSLATION = False
TRANSLATE_JSONL_IN = WORK_DIR / "zolai_cleaner_out" / "zolai_train_ready_v2.jsonl"
TRANSLATE_JSONL_OUT = WORK_DIR / "zolai_cleaner_out" / "zolai_with_translation_en.jsonl"
TRANSLATE_BATCH = 32
GEMINI_MODEL = os.environ.get("ZOLOAI_GEMINI_MODEL", "gemini-2.0-flash")


def _extract_json_array(s: str):
    if not s:
        return None
    start = s.find("[")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(s)):
        if s[i] == "[":
            depth += 1
        elif s[i] == "]":
            depth -= 1
            if depth == 0:
                return s[start : i + 1]
    return None


def run_gemini_translation():
    if not RUN_TRANSLATION:
        print("Set RUN_TRANSLATION = True to run.")
        return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai>=1.0.0"])
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            u = UserSecretsClient()
            key = u.get_secret("GEMINI_API_KEY") or u.get_secret("GOOGLE_API_KEY")
        except Exception:
            key = None
    if not key:
        raise RuntimeError("Need GEMINI_API_KEY or GOOGLE_API_KEY")
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=key)
    texts = []
    with open(TRANSLATE_JSONL_IN, "r", encoding="utf-8") as f:
        for line in f:
            o = json.loads(line)
            t = (o.get("text") or "").strip()
            if not t:
                continue
            te = (o.get("translation_en") or "").strip()
            if te:
                continue
            texts.append((o, t))
            if len(texts) >= TRANSLATE_BATCH:
                break
    if not texts:
        print("No rows needing translation (or file empty).")
        return
    batch = [x[1] for x in texts]
    joined = "\n".join(f"{i+1}. {x}" for i, x in enumerate(batch))
    prompt = (
        "Translate each line to English. Return ONLY a JSON array of strings, "
        "same length and order as inputs.\n\n"
        f"INPUTS:\n{joined}"
    )
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[types.Content(role="user", parts=[types.Part(text=prompt)])],
        config=types.GenerateContentConfig(temperature=0.0),
    )
    raw = (resp.text or "").strip()
    arr = json.loads(_extract_json_array(raw) or raw)
    if len(arr) != len(batch):
        raise ValueError("Length mismatch")
    for (obj, _), en in zip(texts, arr):
        obj["translation_en"] = str(en).strip()
    with open(TRANSLATE_JSONL_OUT, "w", encoding="utf-8") as fout:
        for obj, _ in texts:
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
    print("Wrote sample batch to", TRANSLATE_JSONL_OUT, "— extend loop for full corpus.")


run_gemini_translation()


## 4) Why not one giant Run All?

- **Cleaner + combiner + audit** can approach **hours** and **many GB** on large JSONL.
- **Gemma** + **Qwen** in same notebook = **two heavy GPU** phases; Kaggle often better as **two GPU jobs**.
- **Failures** are easier to debug when **modules** are separate.

Use this notebook as **config + translation helper + checklist**; keep **`zolai-cleaner-v2.ipynb`** etc. as the **source of truth** for the main steps.
